# Building scenario feeds

A **scenario feed** is a GTFS (General Transit Feed Specification) feed for
a planned public transport network, for example to compare travel times with
and without a new tram line. With `build_feed`, you can turn routes drawn in
any GIS tool, together with their headways (how often they run), into a
validated GTFS feed. Let's draw two routes across the centre of Turku as a
`GeoDataFrame` and save them to a GeoPackage, as a GIS tool would:

In [1]:
import geopandas as gpd
from shapely.geometry import LineString

routes = gpd.GeoDataFrame(
    {
        "route_short_name": ["T1", "B2"],
        "mode": ["tram", "bus"],
        "headway_min": [10, 15],
        "speed_kmh": [20, 25],
    },
    geometry=[
        LineString([(22.235, 60.452), (22.268, 60.452), (22.300, 60.445)]),
        LineString([(22.250, 60.440), (22.267, 60.452), (22.280, 60.468)]),
    ],
    crs="EPSG:4326",
)
routes.to_file("planned_network.gpkg", layer="routes")
routes

,route_short_name,mode,headway_min,speed_kmh,geometry
0,T1,tram,10,20,"LINESTRING (22.235 60.452, 22.268 60.452, 22.3..."
1,B2,bus,15,25,"LINESTRING (22.25 60.44, 22.267 60.452, 22.28 ..."


`build_feed` reads the route layer and writes the feed:

In [2]:
import transitio

report = transitio.build_feed(
    "planned_network.gpkg",
    "scenario.zip",
    routes_layer="routes",        # LineStrings with attributes:
    timezone="Europe/Helsinki",   #   mode, headway_min, speed_kmh, days...
)
report["row_counts"]

{'agency.txt': 1,
 'calendar.txt': 2,
 'frequencies.txt': 4,
 'routes.txt': 2,
 'shapes.txt': 12,
 'stop_times.txt': 40,
 'stops.txt': 20,
 'trips.txt': 4}

Stops come from an optional point layer (`stops_layer=`), snapped to each
route, or are placed along each route every `stop_spacing` metres (400 m by
default). The route geometries become GTFS shapes, so routing keeps the
travel distances along the drawn routes. You can pass the result straight to
`cafein.TransportNetwork.from_gtfs`:

In [3]:
import cafein

network = cafein.TransportNetwork.from_gtfs(["scenario.zip"])
print(network.stop_count, "stops,", network.trip_count, "trips")

20 stops, 320 trips
